# DRW + Neural Anomaly Detection — DataViz, statistiques modèle, XAI

Ce notebook complète le notebook d'entraînement précédent avec :

1. **DataViz réelles**
   - distribution du nombre de points par source ;
   - distribution du flux, des erreurs, des gaps temporels ;
   - SNR robuste ;
   - visualisation de sources aléatoires.

2. **DataViz synthétiques**
   - light curves DRW simulées ;
   - outliers injectés ;
   - distributions des labels et features.

3. **Statistiques modèle**
   - loss / F1 / precision / recall ;
   - ROC-AUC / PR-AUC ;
   - courbes ROC et Precision-Recall ;
   - matrice de confusion ;
   - sweep du seuil ;
   - calibration des probabilités ;
   - métriques par source.

4. **XAI**
   - permutation importance globale ;
   - ablation de features ;
   - gradients/saliency ;
   - integrated gradients ;
   - occlusion temporelle ;
   - explication point par point sur une vraie source.

Le modèle visé est le même :
- simulation de quasars par **DRW / Ornstein-Uhlenbeck** ;
- séquences irrégulières ;
- features temporelles ;
- **BiGRU** point-wise ;
- sortie : probabilité que chaque point soit un outlier.

In [ ]:
# ============================================================
# 0. Imports
# ============================================================

# Si nécessaire :
# !pip install torch scikit-learn tqdm

from pathlib import Path
import math
import random
import warnings
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        precision_recall_fscore_support,
        confusion_matrix,
        roc_curve,
        precision_recall_curve,
        brier_score_loss,
    )
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

## 1. Configuration

Par défaut, le notebook tente de charger le modèle sauvegardé :

```text
models/drw_bigru_point_anomaly_detector.pt
```

S'il n'existe pas, il entraîne un nouveau modèle.

In [ ]:
# ============================================================
# 1. Config
# ============================================================

ROOT = Path.cwd()

DATA_PATH = ROOT / "data" / "EOLENS_2_6.csv"
MODEL_PATH = ROOT / "models" / "drw_bigru_point_anomaly_detector.pt"

SOURCE_COL = "source_id"
COMPONENT_COL = "lensComponentSourceId"
Y_COL = "flux_obs"
YERR_COL = "flux_obs_error"

MAX_SOURCES_REAL = 2000

# Training config si le modèle n'existe pas
TRAIN_IF_MODEL_MISSING = True
TRAIN_SEQUENCES = 8000
VAL_SEQUENCES = 1500
EPOCHS = 10
BATCH_SIZE = 128

# Inference / plots
MAX_PLOTS = 20

## 2. Chargement des données réelles

In [ ]:
# ============================================================
# 2. Load real data
# ============================================================

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Data shape:", df.shape)
display(df.head())
print(df.columns.tolist())


def guess_time_col(df):
    candidates = [
        "epoch_obs_jd",
        "jd_time", "jdTime", "JD_TIME",
        "jd", "JD",
        "mjd", "MJD",
        "julian_date", "JulianDate", "julianDate",
        "time", "Time",
        "timestamp", "Timestamp",
        "datetime", "Datetime",
        "date", "Date",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    for c in df.columns:
        name = str(c).lower()
        if "jd" in name or "julian" in name or "time" in name or "date" in name or "epoch" in name:
            return c

    raise ValueError("No time column found")


TIME_COL = guess_time_col(df)

print("TIME_COL      =", TIME_COL)
print("SOURCE_COL    =", SOURCE_COL)
print("COMPONENT_COL =", COMPONENT_COL)
print("Y_COL         =", Y_COL)
print("YERR_COL      =", YERR_COL)

## 3. Fonctions utilitaires

In [ ]:
# ============================================================
# 3. Utils
# ============================================================

def time_to_float_days(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() >= max(3, int(0.5 * len(series))):
        return numeric.to_numpy(dtype=float)

    dt = pd.to_datetime(series, utc=True, errors="coerce")
    arr = dt.astype("int64").to_numpy(dtype=float)

    nat_value = np.iinfo("int64").min
    arr[arr == nat_value] = np.nan

    return arr / 1e9 / 86400.0


def robust_sigma(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) < 3:
        return np.nan

    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    sig = 1.4826 * mad

    if not np.isfinite(sig) or sig <= 0:
        sig = np.nanstd(x)

    return float(sig)


def safe_yerr(yerr):
    yerr = np.asarray(yerr, dtype=float)
    good = np.isfinite(yerr) & (yerr > 0)

    if good.any():
        fallback = np.nanmedian(yerr[good])
    else:
        fallback = 1.0

    if not np.isfinite(fallback) or fallback <= 0:
        fallback = 1.0

    return np.where(good, yerr, fallback)


def get_group_cols(df):
    cols = []

    if SOURCE_COL in df.columns:
        cols.append(SOURCE_COL)

    if COMPONENT_COL in df.columns:
        cols.append(COMPONENT_COL)

    if not cols:
        raise ValueError("No source/component columns found")

    return cols


GROUP_COLS = get_group_cols(df)


def group_title(keys, group_cols=GROUP_COLS):
    if not isinstance(keys, tuple):
        keys = (keys,)
    return ", ".join(f"{c}={v}" for c, v in zip(group_cols, keys))


def compute_source_basic_stats(df):
    rows = []

    for keys, sub in tqdm(df.groupby(GROUP_COLS, sort=False), desc="Source stats"):
        t = time_to_float_days(sub[TIME_COL])
        y = pd.to_numeric(sub[Y_COL], errors="coerce").to_numpy(dtype=float)
        yerr = safe_yerr(pd.to_numeric(sub[YERR_COL], errors="coerce").to_numpy(dtype=float))

        finite = np.isfinite(t) & np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)

        if finite.sum() == 0:
            continue

        tt = np.sort(t[finite])
        yy = y[finite]
        ee = yerr[finite]

        gaps = np.diff(tt)
        gaps = gaps[np.isfinite(gaps) & (gaps > 0)]

        q05, q95 = np.nanpercentile(yy, [5, 95]) if len(yy) >= 3 else (np.nan, np.nan)
        signal = 0.5 * (q95 - q05) if np.isfinite(q05) and np.isfinite(q95) else np.nan
        noise = np.nanmedian(ee)
        snr = signal / noise if np.isfinite(signal) and np.isfinite(noise) and noise > 0 else np.nan

        rows.append({
            "group": group_title(keys),
            "n_points": int(finite.sum()),
            "time_span": float(np.nanmax(tt) - np.nanmin(tt)) if len(tt) > 1 else 0.0,
            "median_gap": float(np.nanmedian(gaps)) if len(gaps) > 0 else np.nan,
            "flux_median": float(np.nanmedian(yy)),
            "flux_robust_sigma": robust_sigma(yy),
            "err_median": float(np.nanmedian(ee)),
            "snr_robust_amp": float(snr) if np.isfinite(snr) else np.nan,
        })

    return pd.DataFrame(rows)

# Partie A — DataViz sur les données réelles

In [ ]:
# ============================================================
# 4. DataViz real data summary
# ============================================================

# Pour aller plus vite en notebook, on limite les stats aux premières sources.
unique_groups = df[GROUP_COLS].drop_duplicates().head(MAX_SOURCES_REAL)
df_vis = df.merge(unique_groups, on=GROUP_COLS, how="inner").copy()

source_stats = compute_source_basic_stats(df_vis)

display(source_stats.head())
display(source_stats.describe())

In [ ]:
# ============================================================
# 5. Plots distributions réelles
# ============================================================

def hist_plot(values, title, xlabel, bins=50, logy=False):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(values, bins=bins)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("count")

    if logy:
        ax.set_yscale("log")

    ax.grid(alpha=0.3)
    plt.show()


hist_plot(source_stats["n_points"], "Nombre de points par source", "n_points", bins=50, logy=True)
hist_plot(source_stats["time_span"], "Baseline temporelle par source", "time span [days]", bins=50, logy=False)
hist_plot(source_stats["median_gap"], "Gaps temporels médians", "median gap [days]", bins=50, logy=True)
hist_plot(source_stats["err_median"], "Erreur photométrique médiane", "median flux_obs_error", bins=50, logy=True)
hist_plot(source_stats["snr_robust_amp"], "SNR robuste par source", "robust amplitude / median error", bins=50, logy=True)

In [ ]:
# ============================================================
# 6. Scatter stats source-level
# ============================================================

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    source_stats["n_points"],
    source_stats["snr_robust_amp"],
    s=12,
    alpha=0.5,
)
ax.set_xlabel("n_points")
ax.set_ylabel("robust SNR")
ax.set_title("SNR robuste vs nombre de points")
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    source_stats["err_median"],
    source_stats["flux_robust_sigma"],
    s=12,
    alpha=0.5,
)
ax.set_xlabel("median flux error")
ax.set_ylabel("robust flux sigma")
ax.set_title("Variabilité observée vs erreur photométrique")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# 7. Visualiser quelques sources aléatoires
# ============================================================

def plot_random_real_sources(df, n_sources=6, random_state=42):
    rng = np.random.default_rng(random_state)

    groups = df[GROUP_COLS].drop_duplicates()
    chosen = groups.sample(n=min(n_sources, len(groups)), random_state=random_state)

    for _, row in chosen.iterrows():
        mask = np.ones(len(df), dtype=bool)
        for col in GROUP_COLS:
            mask &= df[col].to_numpy() == row[col]

        sub = df.loc[mask].copy()

        t = time_to_float_days(sub[TIME_COL])
        y = pd.to_numeric(sub[Y_COL], errors="coerce").to_numpy(dtype=float)
        yerr = safe_yerr(pd.to_numeric(sub[YERR_COL], errors="coerce").to_numpy(dtype=float))

        finite = np.isfinite(t) & np.isfinite(y)

        order = np.argsort(t[finite])

        title = ", ".join(f"{c}={row[c]}" for c in GROUP_COLS)

        fig, ax = plt.subplots(figsize=(12, 4))
        ax.errorbar(
            t[finite][order],
            y[finite][order],
            yerr=yerr[finite][order],
            fmt="o",
            ms=4,
            capsize=2,
        )
        ax.set_title(title)
        ax.set_xlabel(f"{TIME_COL} converted to days")
        ax.set_ylabel(Y_COL)
        ax.grid(alpha=0.3)
        plt.show()


plot_random_real_sources(df_vis, n_sources=6, random_state=10)

# Partie B — Simulation DRW et features

In [ ]:
# ============================================================
# 8. Simulation DRW + outliers
# ============================================================

def simulate_irregular_times(
    n_points,
    baseline_days=1200.0,
    seasonal=True,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng()

    if not seasonal:
        return np.sort(rng.uniform(0, baseline_days, size=n_points))

    n_seasons = rng.integers(3, 7)
    season_centers = np.sort(rng.uniform(0, baseline_days, size=n_seasons))
    season_width = rng.uniform(20, 90)

    times = []

    for _ in range(n_points):
        c = rng.choice(season_centers)
        val = rng.normal(c, season_width / 2)
        val = np.clip(val, 0, baseline_days)
        times.append(val)

    t = np.sort(np.array(times, dtype=float))
    t += rng.normal(0, 1e-3, size=len(t))

    return np.sort(t)


def simulate_drw_ou(t, tau, sf_inf, mu=0.0, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    t = np.asarray(t, dtype=float)
    n = len(t)

    sigma_stationary = sf_inf / np.sqrt(2.0)

    x = np.zeros(n, dtype=float)
    x[0] = rng.normal(mu, sigma_stationary)

    for i in range(1, n):
        dt = max(float(t[i] - t[i - 1]), 0.0)
        phi = np.exp(-dt / tau)
        innovation_std = sigma_stationary * np.sqrt(max(1.0 - phi ** 2, 0.0))
        x[i] = mu + phi * (x[i - 1] - mu) + innovation_std * rng.normal()

    return x


def inject_outliers(
    y_true,
    y_obs,
    yerr,
    outlier_fraction,
    rng=None,
    mode="mixed",
):
    if rng is None:
        rng = np.random.default_rng()

    y_true = np.asarray(y_true, dtype=float)
    y_obs = np.asarray(y_obs, dtype=float).copy()
    yerr = np.asarray(yerr, dtype=float).copy()

    n = len(y_obs)
    labels = np.zeros(n, dtype=np.float32)

    n_out = int(round(outlier_fraction * n))
    n_out = min(max(n_out, 0), max(n - 1, 0))

    if n_out == 0:
        return y_obs, yerr, labels

    out_idx = rng.choice(n, size=n_out, replace=False)
    labels[out_idx] = 1.0

    base_scale = robust_sigma(y_true)

    if not np.isfinite(base_scale) or base_scale <= 0:
        base_scale = np.nanstd(y_true)

    if not np.isfinite(base_scale) or base_scale <= 0:
        base_scale = np.nanmedian(yerr)

    if not np.isfinite(base_scale) or base_scale <= 0:
        base_scale = 1.0

    for idx in out_idx:
        kind = rng.choice(["spike", "level", "bad_error"], p=[0.65, 0.25, 0.10])

        sign = rng.choice([-1.0, 1.0])
        amp = rng.uniform(4.0, 10.0) * np.sqrt(yerr[idx] ** 2 + base_scale ** 2)

        if mode == "positive":
            sign = 1.0

        if kind == "spike":
            y_obs[idx] += sign * amp

        elif kind == "level":
            y_obs[idx] += sign * rng.uniform(2.5, 6.0) * base_scale

        elif kind == "bad_error":
            y_obs[idx] += sign * rng.uniform(3.0, 8.0) * yerr[idx]
            yerr[idx] *= rng.uniform(0.2, 0.7)

    return y_obs, yerr, labels


def simulate_one_quasar_sequence(
    min_points=25,
    max_points=60,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng()

    n_points = int(rng.integers(min_points, max_points + 1))
    baseline = rng.uniform(300, 2500)

    t = simulate_irregular_times(
        n_points,
        baseline_days=baseline,
        seasonal=True,
        rng=rng,
    )

    tau = 10 ** rng.uniform(np.log10(80), np.log10(800))
    sf_inf = rng.uniform(20, 120)
    mu = rng.normal(100, 30)

    y_true = simulate_drw_ou(
        t,
        tau=tau,
        sf_inf=sf_inf,
        mu=mu,
        rng=rng,
    )

    median_err = rng.uniform(3, 20)
    yerr = median_err * np.exp(rng.normal(0, 0.35, size=n_points))
    y_obs = y_true + rng.normal(0, yerr)

    outlier_fraction = rng.uniform(0.00, 0.18)

    y_obs, yerr, labels = inject_outliers(
        y_true,
        y_obs,
        yerr,
        outlier_fraction=outlier_fraction,
        rng=rng,
        mode="mixed",
    )

    return {
        "t": t.astype(np.float32),
        "y": y_obs.astype(np.float32),
        "yerr": yerr.astype(np.float32),
        "label": labels.astype(np.float32),
        "y_true": y_true.astype(np.float32),
        "tau": tau,
        "sf_inf": sf_inf,
    }

In [ ]:
# ============================================================
# 9. Visualiser simulations DRW
# ============================================================

def plot_synthetic_examples(n=4, seed=123):
    rng = np.random.default_rng(seed)

    for i in range(n):
        seq = simulate_one_quasar_sequence(rng=rng)

        t = seq["t"]
        y = seq["y"]
        yerr = seq["yerr"]
        ytrue = seq["y_true"]
        lab = seq["label"].astype(bool)

        order = np.argsort(t)

        fig, ax = plt.subplots(figsize=(12, 4))

        ax.plot(t[order], ytrue[order], "-", linewidth=2, alpha=0.7, label="latent DRW")
        ax.errorbar(
            t[~lab],
            y[~lab],
            yerr=yerr[~lab],
            fmt="o",
            ms=4,
            capsize=2,
            label="normal obs",
        )
        ax.errorbar(
            t[lab],
            y[lab],
            yerr=yerr[lab],
            fmt="x",
            ms=8,
            capsize=2,
            label="injected outlier",
        )

        ax.set_title(f"Synthetic DRW example | tau={seq['tau']:.1f}, sf_inf={seq['sf_inf']:.1f}")
        ax.set_xlabel("time [days]")
        ax.set_ylabel("flux")
        ax.grid(alpha=0.3)
        ax.legend()
        plt.show()


plot_synthetic_examples(n=4, seed=5)

In [ ]:
# ============================================================
# 10. Features
# ============================================================

FEATURE_NAMES = [
    "y_scaled",
    "log_err_scaled",
    "dt_prev",
    "dt_next",
    "local_resid_scaled",
    "local_z",
    "abs_y_scaled",
]


def loo_local_features(t, y, yerr, k_neighbors=8, min_neighbors=4):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    yerr = safe_yerr(yerr)

    n = len(y)

    local_med = np.full(n, np.nan)
    local_z = np.full(n, np.nan)

    finite = np.isfinite(t) & np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)

    for i in range(n):
        if not finite[i]:
            continue

        candidates = np.where(finite)[0]
        candidates = candidates[candidates != i]

        if len(candidates) < min_neighbors:
            continue

        dist = np.abs(t[candidates] - t[i])

        if len(candidates) > k_neighbors:
            neigh = candidates[np.argsort(dist)[:k_neighbors]]
        else:
            neigh = candidates

        if len(neigh) < min_neighbors:
            continue

        yy = y[neigh]
        med = np.nanmedian(yy)
        scat = robust_sigma(yy)

        err_med = np.nanmedian(yerr[neigh])

        if not np.isfinite(err_med) or err_med <= 0:
            err_med = np.nanmedian(yerr[finite])

        if not np.isfinite(err_med) or err_med <= 0:
            err_med = 1.0

        if not np.isfinite(scat) or scat <= 0:
            scat = err_med

        scat = max(scat, err_med)
        scat = min(scat, 8.0 * err_med)

        total = np.sqrt(yerr[i] ** 2 + scat ** 2)
        total = max(total, 1e-12)

        local_med[i] = med
        local_z[i] = (y[i] - med) / total

    return local_med, local_z


def make_sequence_features(t, y, yerr, max_abs_z_clip=20.0):
    t = np.asarray(t, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    yerr = safe_yerr(np.asarray(yerr, dtype=np.float32)).astype(np.float32)

    order = np.argsort(t)

    t_sorted_all = t[order]
    y_sorted_all = y[order]
    yerr_sorted_all = yerr[order]

    finite = (
        np.isfinite(t_sorted_all)
        & np.isfinite(y_sorted_all)
        & np.isfinite(yerr_sorted_all)
        & (yerr_sorted_all > 0)
    )

    t = t_sorted_all[finite]
    y = y_sorted_all[finite]
    yerr = yerr_sorted_all[finite]

    n = len(y)

    if n == 0:
        return np.zeros((0, len(FEATURE_NAMES)), dtype=np.float32), order, finite

    y_med = np.nanmedian(y)
    y_sig = robust_sigma(y)

    if not np.isfinite(y_sig) or y_sig <= 0:
        y_sig = np.nanstd(y)

    if not np.isfinite(y_sig) or y_sig <= 0:
        y_sig = np.nanmedian(yerr)

    if not np.isfinite(y_sig) or y_sig <= 0:
        y_sig = 1.0

    y_scaled = (y - y_med) / y_sig
    err_scaled = yerr / y_sig
    log_err_scaled = np.log1p(err_scaled)

    dt = np.diff(t)
    dt_pos = dt[np.isfinite(dt) & (dt > 0)]
    dt_med = np.nanmedian(dt_pos) if len(dt_pos) else 1.0

    if not np.isfinite(dt_med) or dt_med <= 0:
        dt_med = 1.0

    dt_prev = np.zeros(n, dtype=np.float32)
    dt_next = np.zeros(n, dtype=np.float32)

    if n > 1:
        dt_prev[1:] = np.diff(t) / dt_med
        dt_next[:-1] = np.diff(t) / dt_med

    dt_prev = np.log1p(np.clip(dt_prev, 0, 1e6))
    dt_next = np.log1p(np.clip(dt_next, 0, 1e6))

    local_med, local_z = loo_local_features(t, y, yerr, k_neighbors=8, min_neighbors=4)

    local_resid_scaled = np.zeros(n, dtype=np.float32)
    ok_med = np.isfinite(local_med)
    local_resid_scaled[ok_med] = (y[ok_med] - local_med[ok_med]) / y_sig

    local_z_clean = np.nan_to_num(
        local_z,
        nan=0.0,
        posinf=max_abs_z_clip,
        neginf=-max_abs_z_clip,
    )
    local_z_clean = np.clip(local_z_clean, -max_abs_z_clip, max_abs_z_clip)

    abs_y_scaled = np.abs(y_scaled)

    X = np.stack(
        [
            y_scaled,
            log_err_scaled,
            dt_prev,
            dt_next,
            local_resid_scaled,
            local_z_clean,
            abs_y_scaled,
        ],
        axis=1,
    ).astype(np.float32)

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    return X, order, finite

In [ ]:
# ============================================================
# 11. DataViz features synthétiques
# ============================================================

def collect_synthetic_feature_table(n_sequences=1000, seed=456):
    rows = []
    rng_master = np.random.default_rng(seed)

    for i in tqdm(range(n_sequences), desc="Collect synthetic features"):
        rng = np.random.default_rng(seed + i)
        seq = simulate_one_quasar_sequence(rng=rng)

        X, order, finite = make_sequence_features(seq["t"], seq["y"], seq["yerr"])
        labels = np.asarray(seq["label"])[order][finite]

        for j in range(len(X)):
            row = {name: X[j, k] for k, name in enumerate(FEATURE_NAMES)}
            row["label"] = int(labels[j])
            rows.append(row)

    return pd.DataFrame(rows)


synthetic_features_df = collect_synthetic_feature_table(n_sequences=1000, seed=500)

display(synthetic_features_df.head())
display(synthetic_features_df.groupby("label").mean())

for name in FEATURE_NAMES:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(synthetic_features_df.loc[synthetic_features_df["label"] == 0, name], bins=60, alpha=0.6, label="normal", density=True)
    ax.hist(synthetic_features_df.loc[synthetic_features_df["label"] == 1, name], bins=60, alpha=0.6, label="outlier", density=True)
    ax.set_title(f"Feature distribution: {name}")
    ax.set_xlabel(name)
    ax.set_ylabel("density")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

# Partie C — Modèle neural

In [ ]:
# ============================================================
# 12. Dataset + model
# ============================================================

class SyntheticDRWOutlierDataset(Dataset):
    def __init__(
        self,
        n_sequences,
        min_points=25,
        max_points=60,
        seed=0,
    ):
        self.n_sequences = int(n_sequences)
        self.min_points = int(min_points)
        self.max_points = int(max_points)
        self.seed = int(seed)

    def __len__(self):
        return self.n_sequences

    def __getitem__(self, idx):
        rng = np.random.default_rng(self.seed + idx)

        seq = simulate_one_quasar_sequence(
            min_points=self.min_points,
            max_points=self.max_points,
            rng=rng,
        )

        X, order, finite = make_sequence_features(seq["t"], seq["y"], seq["yerr"])

        labels = np.asarray(seq["label"], dtype=np.float32)
        labels_sorted = labels[order][finite]

        return {
            "X": torch.tensor(X, dtype=torch.float32),
            "y": torch.tensor(labels_sorted, dtype=torch.float32),
        }


def collate_padded(batch):
    lengths = [item["X"].shape[0] for item in batch]
    max_len = max(lengths)
    feat_dim = batch[0]["X"].shape[1]

    X = torch.zeros(len(batch), max_len, feat_dim, dtype=torch.float32)
    y = torch.zeros(len(batch), max_len, dtype=torch.float32)
    mask = torch.zeros(len(batch), max_len, dtype=torch.bool)

    for b, item in enumerate(batch):
        n = item["X"].shape[0]
        X[b, :n] = item["X"]
        y[b, :n] = item["y"]
        mask[b, :n] = True

    return X, y, mask


class BiGRUPointAnomalyDetector(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim=96,
        num_layers=2,
        dropout=0.15,
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.head = nn.Sequential(
            nn.Linear(2 * hidden_dim + hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, X, mask=None):
        Z = self.input_proj(X)
        H, _ = self.gru(Z)
        joined = torch.cat([Z, H], dim=-1)
        logits = self.head(joined).squeeze(-1)

        if mask is not None:
            logits = logits.masked_fill(~mask, 0.0)

        return logits


model = BiGRUPointAnomalyDetector(input_dim=len(FEATURE_NAMES)).to(DEVICE)

print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# ============================================================
# 13. Load model if exists or train
# ============================================================

train_ds = SyntheticDRWOutlierDataset(TRAIN_SEQUENCES, seed=100)
val_ds = SyntheticDRWOutlierDataset(VAL_SEQUENCES, seed=100000)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_padded,
)

val_loader = DataLoader(
    val_ds,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_padded,
)


def masked_bce_with_logits(logits, targets, mask, pos_weight=None):
    loss_fn = nn.BCEWithLogitsLoss(
        reduction="none",
        pos_weight=pos_weight,
    )
    loss = loss_fn(logits, targets)
    loss = loss[mask]
    return loss.mean()


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()

    all_probs = []
    all_labels = []

    for X, y, mask in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        mask = mask.to(DEVICE)

        logits = model(X, mask)
        probs = torch.sigmoid(logits)

        all_probs.append(probs[mask].detach().cpu().numpy())
        all_labels.append(y[mask].detach().cpu().numpy())

    return np.concatenate(all_probs), np.concatenate(all_labels)


def metric_dict_from_probs(probs, labels, threshold=0.5):
    pred = (probs >= threshold).astype(int)
    labels_int = labels.astype(int)

    out = {
        "threshold": threshold,
        "positive_rate_true": float(labels.mean()),
        "positive_rate_pred": float(pred.mean()),
    }

    if SKLEARN_AVAILABLE:
        try:
            out["roc_auc"] = float(roc_auc_score(labels_int, probs))
        except Exception:
            out["roc_auc"] = np.nan

        try:
            out["pr_auc"] = float(average_precision_score(labels_int, probs))
        except Exception:
            out["pr_auc"] = np.nan

        p, r, f1, _ = precision_recall_fscore_support(
            labels_int,
            pred,
            average="binary",
            zero_division=0,
        )

        out["precision"] = float(p)
        out["recall"] = float(r)
        out["f1"] = float(f1)

        cm = confusion_matrix(labels_int, pred)
        out["tn"] = int(cm[0, 0]) if cm.shape == (2, 2) else np.nan
        out["fp"] = int(cm[0, 1]) if cm.shape == (2, 2) else np.nan
        out["fn"] = int(cm[1, 0]) if cm.shape == (2, 2) else np.nan
        out["tp"] = int(cm[1, 1]) if cm.shape == (2, 2) else np.nan

        try:
            out["brier"] = float(brier_score_loss(labels_int, probs))
        except Exception:
            out["brier"] = np.nan

    return out


def find_best_threshold_from_probs(probs, labels, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 91)

    labels_int = labels.astype(int)

    best = None
    rows = []

    for th in thresholds:
        pred = (probs >= th).astype(int)

        if SKLEARN_AVAILABLE:
            p, r, f1, _ = precision_recall_fscore_support(
                labels_int,
                pred,
                average="binary",
                zero_division=0,
            )
        else:
            tp = np.sum((pred == 1) & (labels_int == 1))
            fp = np.sum((pred == 1) & (labels_int == 0))
            fn = np.sum((pred == 0) & (labels_int == 1))
            p = tp / max(tp + fp, 1)
            r = tp / max(tp + fn, 1)
            f1 = 2 * p * r / max(p + r, 1e-12)

        row = {"threshold": float(th), "precision": float(p), "recall": float(r), "f1": float(f1)}
        rows.append(row)

        if best is None or row["f1"] > best["f1"]:
            best = row

    return best, pd.DataFrame(rows)


def train_model_if_needed():
    global model

    history = []

    Xb, yb, mb = next(iter(train_loader))
    pos_rate_est = float(yb[mb].mean().item())
    pos_weight_value = (1.0 - pos_rate_est) / max(pos_rate_est, 1e-4)
    pos_weight_value = min(pos_weight_value, 25.0)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for X, y, mask in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            X = X.to(DEVICE)
            y = y.to(DEVICE)
            mask = mask.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(X, mask)
            loss = masked_bce_with_logits(logits, y, mask, pos_weight=pos_weight)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            losses.append(float(loss.item()))

        scheduler.step()

        probs, labels = collect_predictions(model, val_loader)
        best, _ = find_best_threshold_from_probs(probs, labels)
        metrics = metric_dict_from_probs(probs, labels, threshold=best["threshold"])
        metrics["epoch"] = epoch
        metrics["train_loss"] = float(np.mean(losses))
        history.append(metrics)

        print(
            f"epoch={epoch:02d} "
            f"loss={metrics['train_loss']:.4f} "
            f"th={metrics['threshold']:.2f} "
            f"P={metrics['precision']:.3f} "
            f"R={metrics['recall']:.3f} "
            f"F1={metrics['f1']:.3f} "
            f"PR_AUC={metrics.get('pr_auc', np.nan):.3f}"
        )

    best_threshold = float(pd.DataFrame(history).sort_values("f1", ascending=False).iloc[0]["threshold"])

    MODEL_PATH.parent.mkdir(exist_ok=True)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "input_dim": len(FEATURE_NAMES),
            "best_threshold": best_threshold,
            "history": history,
            "feature_names": FEATURE_NAMES,
        },
        MODEL_PATH,
    )

    return history, best_threshold


if MODEL_PATH.exists():
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    best_threshold = float(ckpt.get("best_threshold", 0.5))
    history = ckpt.get("history", [])
    print("Loaded model:", MODEL_PATH)
    print("best_threshold:", best_threshold)

else:
    if not TRAIN_IF_MODEL_MISSING:
        raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

    print("Model not found, training a new model...")
    history, best_threshold = train_model_if_needed()
    print("Saved model:", MODEL_PATH)

# Partie D — Statistiques du modèle

In [ ]:
# ============================================================
# 14. Validation metrics
# ============================================================

val_probs, val_labels = collect_predictions(model, val_loader)
best_threshold_row, threshold_sweep_df = find_best_threshold_from_probs(val_probs, val_labels)

best_threshold = float(best_threshold_row["threshold"])

metrics = metric_dict_from_probs(val_probs, val_labels, threshold=best_threshold)

print("Best threshold from validation:", best_threshold)
display(pd.DataFrame([metrics]))

display(threshold_sweep_df.sort_values("f1", ascending=False).head(10))

In [ ]:
# ============================================================
# 15. Training history plots if available
# ============================================================

if history is not None and len(history) > 0:
    history_df = pd.DataFrame(history)
    display(history_df)

    for col in ["train_loss", "precision", "recall", "f1", "pr_auc", "roc_auc"]:
        if col in history_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(history_df["epoch"], history_df[col], marker="o")
            ax.set_title(f"Training history: {col}")
            ax.set_xlabel("epoch")
            ax.set_ylabel(col)
            ax.grid(alpha=0.3)
            plt.show()
else:
    print("No training history found in checkpoint.")

In [ ]:
# ============================================================
# 16. ROC, PR, confusion, threshold sweep, calibration
# ============================================================

if SKLEARN_AVAILABLE:
    fpr, tpr, _ = roc_curve(val_labels.astype(int), val_probs)
    prec, rec, _ = precision_recall_curve(val_labels.astype(int), val_probs)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr)
    ax.plot([0, 1], [0, 1], "--")
    ax.set_title(f"ROC curve | AUC={metrics['roc_auc']:.3f}")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.grid(alpha=0.3)
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(rec, prec)
    ax.set_title(f"Precision-Recall curve | AP={metrics['pr_auc']:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.grid(alpha=0.3)
    plt.show()

    pred = (val_probs >= best_threshold).astype(int)
    cm = confusion_matrix(val_labels.astype(int), pred)

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_title("Confusion matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["normal", "outlier"])
    ax.set_yticklabels(["normal", "outlier"])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.colorbar(im, ax=ax)
    plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], label="precision")
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], label="recall")
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], label="f1")
ax.axvline(best_threshold, linestyle="--", label=f"best th={best_threshold:.2f}")
ax.set_title("Threshold sweep")
ax.set_xlabel("threshold")
ax.set_ylabel("metric")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

# Calibration
bins = np.linspace(0, 1, 11)
bin_ids = np.digitize(val_probs, bins) - 1

calib_rows = []
for b in range(len(bins) - 1):
    m = bin_ids == b
    if m.sum() == 0:
        continue
    calib_rows.append({
        "bin_left": bins[b],
        "bin_right": bins[b + 1],
        "mean_proba": float(np.mean(val_probs[m])),
        "true_rate": float(np.mean(val_labels[m])),
        "count": int(m.sum()),
    })

calib_df = pd.DataFrame(calib_rows)
display(calib_df)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", label="perfect calibration")
ax.plot(calib_df["mean_proba"], calib_df["true_rate"], marker="o", label="model")
ax.set_title("Calibration curve")
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("observed outlier rate")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

# Partie E — XAI globale

On calcule l'importance des features par deux méthodes :

1. **Permutation importance**
   - on mélange une feature dans le batch ;
   - si la métrique baisse beaucoup, la feature est importante.

2. **Ablation**
   - on met une feature à zéro ;
   - on observe la baisse de F1 / PR-AUC.

In [ ]:
# ============================================================
# 17. XAI: permutation importance + ablation globale
# ============================================================

@torch.no_grad()
def collect_predictions_with_feature_transform(model, loader, transform_fn=None):
    model.eval()

    all_probs = []
    all_labels = []

    for X, y, mask in loader:
        if transform_fn is not None:
            X = transform_fn(X.clone(), mask.clone())

        X = X.to(DEVICE)
        y = y.to(DEVICE)
        mask = mask.to(DEVICE)

        logits = model(X, mask)
        probs = torch.sigmoid(logits)

        all_probs.append(probs[mask].detach().cpu().numpy())
        all_labels.append(y[mask].detach().cpu().numpy())

    return np.concatenate(all_probs), np.concatenate(all_labels)


def score_probs(probs, labels, threshold):
    m = metric_dict_from_probs(probs, labels, threshold=threshold)
    return {
        "f1": m.get("f1", np.nan),
        "precision": m.get("precision", np.nan),
        "recall": m.get("recall", np.nan),
        "pr_auc": m.get("pr_auc", np.nan),
        "roc_auc": m.get("roc_auc", np.nan),
    }


base_scores = score_probs(val_probs, val_labels, best_threshold)

rows = []

for feat_idx, feat_name in enumerate(FEATURE_NAMES):
    def perm_transform(X, mask, feat_idx=feat_idx):
        vals = X[:, :, feat_idx].clone()
        flat = vals[mask]
        perm = flat[torch.randperm(len(flat))]
        vals[mask] = perm
        X[:, :, feat_idx] = vals
        return X

    p_probs, p_labels = collect_predictions_with_feature_transform(model, val_loader, transform_fn=perm_transform)
    p_scores = score_probs(p_probs, p_labels, best_threshold)

    def zero_transform(X, mask, feat_idx=feat_idx):
        X[:, :, feat_idx] = 0.0
        return X

    z_probs, z_labels = collect_predictions_with_feature_transform(model, val_loader, transform_fn=zero_transform)
    z_scores = score_probs(z_probs, z_labels, best_threshold)

    rows.append({
        "feature": feat_name,
        "base_f1": base_scores["f1"],
        "perm_f1": p_scores["f1"],
        "perm_delta_f1": base_scores["f1"] - p_scores["f1"],
        "zero_f1": z_scores["f1"],
        "zero_delta_f1": base_scores["f1"] - z_scores["f1"],
        "base_pr_auc": base_scores["pr_auc"],
        "perm_pr_auc": p_scores["pr_auc"],
        "perm_delta_pr_auc": base_scores["pr_auc"] - p_scores["pr_auc"],
        "zero_pr_auc": z_scores["pr_auc"],
        "zero_delta_pr_auc": base_scores["pr_auc"] - z_scores["pr_auc"],
    })

xai_global_df = pd.DataFrame(rows).sort_values("perm_delta_f1", ascending=False)
display(xai_global_df)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(xai_global_df["feature"], xai_global_df["perm_delta_f1"])
ax.set_title("Permutation importance — delta F1")
ax.set_xlabel("F1 drop after permutation")
ax.invert_yaxis()
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(xai_global_df["feature"], xai_global_df["zero_delta_f1"])
ax.set_title("Feature ablation — delta F1")
ax.set_xlabel("F1 drop after zeroing feature")
ax.invert_yaxis()
ax.grid(alpha=0.3)
plt.show()

# Partie F — XAI locale : gradients, Integrated Gradients, occlusion

Pour une séquence donnée :
- `gradient saliency` mesure la sensibilité locale de la probabilité à chaque feature ;
- `integrated gradients` accumule cette sensibilité depuis une baseline ;
- `occlusion` masque des points ou des fenêtres temporelles et observe la baisse de probabilité.

In [ ]:
# ============================================================
# 18. XAI local functions
# ============================================================

@torch.no_grad()
def predict_one_sequence(model, t, y, yerr):
    X, order, finite = make_sequence_features(t, y, yerr)

    if len(X) == 0:
        return None

    Xt = torch.tensor(X, dtype=torch.float32)[None].to(DEVICE)
    mask = torch.ones(1, Xt.shape[1], dtype=torch.bool, device=DEVICE)

    logits = model(Xt, mask)
    probs = torch.sigmoid(logits)[0].cpu().numpy()

    t_sorted = np.asarray(t)[order][finite]
    y_sorted = np.asarray(y)[order][finite]
    yerr_sorted = np.asarray(yerr)[order][finite]

    return {
        "X": X,
        "t": t_sorted,
        "y": y_sorted,
        "yerr": yerr_sorted,
        "probs": probs,
        "order": order,
        "finite": finite,
    }


def gradient_saliency(model, X_np, target_index):
    model.eval()

    X = torch.tensor(X_np, dtype=torch.float32, device=DEVICE)[None]
    X.requires_grad_(True)

    mask = torch.ones(1, X.shape[1], dtype=torch.bool, device=DEVICE)

    logits = model(X, mask)
    target_logit = logits[0, target_index]

    model.zero_grad(set_to_none=True)
    target_logit.backward()

    grad = X.grad.detach().cpu().numpy()[0]
    sal = np.abs(grad * X.detach().cpu().numpy()[0])

    return grad, sal


def integrated_gradients(model, X_np, target_index, baseline=None, steps=32):
    model.eval()

    X_np = np.asarray(X_np, dtype=np.float32)

    if baseline is None:
        baseline = np.zeros_like(X_np, dtype=np.float32)
    else:
        baseline = np.asarray(baseline, dtype=np.float32)

    total_grad = np.zeros_like(X_np, dtype=np.float32)

    for alpha in np.linspace(0.0, 1.0, steps):
        X_step_np = baseline + alpha * (X_np - baseline)

        X_step = torch.tensor(X_step_np, dtype=torch.float32, device=DEVICE)[None]
        X_step.requires_grad_(True)

        mask = torch.ones(1, X_step.shape[1], dtype=torch.bool, device=DEVICE)

        logits = model(X_step, mask)
        target_logit = logits[0, target_index]

        model.zero_grad(set_to_none=True)
        target_logit.backward()

        grad = X_step.grad.detach().cpu().numpy()[0]
        total_grad += grad.astype(np.float32)

    avg_grad = total_grad / float(steps)
    ig = (X_np - baseline) * avg_grad

    return ig


@torch.no_grad()
def occlusion_point_importance(model, X_np, target_index, baseline_value=0.0):
    model.eval()

    X = torch.tensor(X_np, dtype=torch.float32, device=DEVICE)[None]
    mask = torch.ones(1, X.shape[1], dtype=torch.bool, device=DEVICE)

    base_prob = torch.sigmoid(model(X, mask))[0, target_index].item()

    drops = np.zeros(X_np.shape[0], dtype=float)

    for i in range(X_np.shape[0]):
        X_occ = X_np.copy()
        X_occ[i, :] = baseline_value

        Xt = torch.tensor(X_occ, dtype=torch.float32, device=DEVICE)[None]
        prob = torch.sigmoid(model(Xt, mask))[0, target_index].item()
        drops[i] = base_prob - prob

    return base_prob, drops


def plot_xai_for_sequence(seq_pred, title="XAI sequence", threshold=None):
    if threshold is None:
        threshold = best_threshold

    t = seq_pred["t"]
    y = seq_pred["y"]
    yerr = seq_pred["yerr"]
    probs = seq_pred["probs"]
    X = seq_pred["X"]

    target_index = int(np.nanargmax(probs))

    grad, sal = gradient_saliency(model, X, target_index=target_index)
    ig = integrated_gradients(model, X, target_index=target_index, steps=32)
    base_prob, occ_drops = occlusion_point_importance(model, X, target_index=target_index)

    feature_sal = sal[target_index]
    feature_ig = np.abs(ig[target_index])

    # Plot light curve + proba
    fig, ax = plt.subplots(figsize=(12, 5))
    is_out = probs >= threshold

    ax.errorbar(
        t[~is_out],
        y[~is_out],
        yerr=yerr[~is_out],
        fmt="o",
        ms=4,
        capsize=2,
        label="pred kept",
    )
    ax.errorbar(
        t[is_out],
        y[is_out],
        yerr=yerr[is_out],
        fmt="x",
        ms=8,
        capsize=2,
        label="pred outlier",
    )

    ax.axvline(t[target_index], linestyle="--", alpha=0.7, label="explained point")

    ax2 = ax.twinx()
    ax2.plot(t, probs, ".", alpha=0.45, label="anomaly proba")
    ax2.axhline(threshold, linestyle="--", alpha=0.5)
    ax2.set_ylabel("anomaly probability")

    ax.set_title(f"{title} | explained index={target_index}, proba={probs[target_index]:.3f}")
    ax.set_xlabel("time")
    ax.set_ylabel("flux")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")
    ax2.legend(loc="upper right")
    plt.show()

    # Feature saliency
    sal_df = pd.DataFrame({
        "feature": FEATURE_NAMES,
        "gradient_x_input_abs": feature_sal,
        "integrated_gradients_abs": feature_ig,
        "raw_feature_value": X[target_index],
    }).sort_values("integrated_gradients_abs", ascending=False)

    display(sal_df)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(sal_df["feature"], sal_df["integrated_gradients_abs"])
    ax.set_title("Local XAI — Integrated gradients by feature")
    ax.set_xlabel("abs contribution")
    ax.invert_yaxis()
    ax.grid(alpha=0.3)
    plt.show()

    # Occlusion importance over time
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t, occ_drops, marker="o")
    ax.axvline(t[target_index], linestyle="--", alpha=0.7)
    ax.set_title("Temporal occlusion importance for explained point")
    ax.set_xlabel("time")
    ax.set_ylabel("probability drop when point is occluded")
    ax.grid(alpha=0.3)
    plt.show()

    return {
        "target_index": target_index,
        "saliency_df": sal_df,
        "occlusion_drops": occ_drops,
        "integrated_gradients": ig,
        "gradient_saliency": sal,
    }

In [ ]:
# ============================================================
# 19. XAI local sur une séquence synthétique
# ============================================================

rng = np.random.default_rng(2025)
seq = simulate_one_quasar_sequence(rng=rng)

seq_pred = predict_one_sequence(model, seq["t"], seq["y"], seq["yerr"])
xai_synthetic = plot_xai_for_sequence(seq_pred, title="Synthetic sequence XAI", threshold=best_threshold)

# Partie G — Inference réelle + stats + XAI sur source réelle

In [ ]:
# ============================================================
# 20. Inference sur vraies sources
# ============================================================

@torch.no_grad()
def predict_group_df(model, sub, threshold, time_col=TIME_COL, y_col=Y_COL, yerr_col=YERR_COL):
    sub = sub.copy()

    t = time_to_float_days(sub[time_col])
    y = pd.to_numeric(sub[y_col], errors="coerce").to_numpy(dtype=float)

    if yerr_col in sub.columns:
        yerr = pd.to_numeric(sub[yerr_col], errors="coerce").to_numpy(dtype=float)
    else:
        yerr = np.ones_like(y)

    X, order, finite = make_sequence_features(t, y, yerr)

    proba_full = np.full(len(sub), np.nan, dtype=float)

    if len(X) > 0:
        Xt = torch.tensor(X, dtype=torch.float32)[None].to(DEVICE)
        mask = torch.ones(1, Xt.shape[1], dtype=torch.bool, device=DEVICE)

        logits = model(Xt, mask)
        probs = torch.sigmoid(logits)[0].cpu().numpy()

        original_positions = np.arange(len(sub))[order][finite]
        proba_full[original_positions] = probs

    sub["_nn_anomaly_proba"] = proba_full
    sub["_nn_threshold"] = threshold
    sub["_nn_is_outlier"] = sub["_nn_anomaly_proba"] >= threshold

    return sub


def plot_nn_group(pred_sub, title, threshold):
    t = time_to_float_days(pred_sub[TIME_COL])
    y = pd.to_numeric(pred_sub[Y_COL], errors="coerce").to_numpy(dtype=float)
    yerr = safe_yerr(pd.to_numeric(pred_sub[YERR_COL], errors="coerce").to_numpy(dtype=float))

    is_out = pred_sub["_nn_is_outlier"].fillna(False).to_numpy(bool)
    proba = pred_sub["_nn_anomaly_proba"].to_numpy(float)

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.errorbar(
        t[~is_out],
        y[~is_out],
        yerr=yerr[~is_out],
        fmt="o",
        ms=4,
        capsize=2,
        alpha=0.75,
        label="kept",
    )

    ax.errorbar(
        t[is_out],
        y[is_out],
        yerr=yerr[is_out],
        fmt="x",
        ms=8,
        capsize=2,
        alpha=0.95,
        label="NN outlier",
    )

    ax2 = ax.twinx()
    ax2.plot(t, proba, ".", alpha=0.35, label="anomaly proba")
    ax2.axhline(threshold, linestyle="--", alpha=0.5)
    ax2.set_ylabel("NN anomaly probability")

    ax.set_title(title)
    ax.set_xlabel(f"{TIME_COL} converted to numeric days")
    ax.set_ylabel(Y_COL)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")
    ax2.legend(loc="upper right")
    plt.show()


def apply_model_to_dataframe(
    model,
    df,
    threshold,
    max_sources=2000,
    source_selection="first",
    min_points_after=20,
    max_outlier_fraction=0.50,
    show_plots=True,
    max_plots=20,
):
    group_cols = get_group_cols(df)

    unique_groups = df[group_cols].drop_duplicates().reset_index(drop=True)

    if max_sources is not None:
        if source_selection == "first":
            selected_groups = unique_groups.head(max_sources)

        elif source_selection == "random":
            selected_groups = unique_groups.sample(
                n=min(max_sources, len(unique_groups)),
                random_state=SEED,
            )

        else:
            raise ValueError("source_selection must be 'first' or 'random'")

    else:
        selected_groups = unique_groups

    work = df.merge(selected_groups, on=group_cols, how="inner").copy()

    parts = []
    reports = []
    plot_count = 0

    for keys, sub in tqdm(work.groupby(group_cols, sort=False), total=len(selected_groups)):
        if not isinstance(keys, tuple):
            keys = (keys,)

        group_name = ", ".join(f"{c}={v}" for c, v in zip(group_cols, keys))

        pred_sub = predict_group_df(model, sub, threshold=threshold)

        n_total = len(pred_sub)
        valid_proba = np.isfinite(pred_sub["_nn_anomaly_proba"].to_numpy())
        n_valid = int(valid_proba.sum())
        n_out = int(pred_sub["_nn_is_outlier"].fillna(False).sum())
        n_keep = int(n_valid - n_out)
        outlier_fraction = n_out / max(n_valid, 1)

        remove_source = False
        reason = "kept_after_nn_cleaning"

        if n_keep < min_points_after:
            remove_source = True
            reason = f"less_than_{min_points_after}_points_after_nn_cleaning"

        elif outlier_fraction > max_outlier_fraction:
            remove_source = True
            reason = f"too_many_nn_outliers_fraction={outlier_fraction:.3f}"

        pred_sub["_nn_remove_source"] = remove_source
        pred_sub["_nn_source_reason"] = reason

        reports.append({
            "group": group_name,
            "removed": remove_source,
            "reason": reason,
            "n_total": n_total,
            "n_valid_pred": n_valid,
            "n_outliers": n_out,
            "n_keep": 0 if remove_source else n_keep,
            "outlier_fraction": outlier_fraction,
            "max_proba": float(np.nanmax(pred_sub["_nn_anomaly_proba"])) if n_valid else np.nan,
            "mean_proba": float(np.nanmean(pred_sub["_nn_anomaly_proba"])) if n_valid else np.nan,
        })

        parts.append(pred_sub)

        if show_plots and n_out > 0 and (max_plots is None or plot_count < max_plots):
            plot_nn_group(pred_sub, group_name, threshold=threshold)
            plot_count += 1

    pred_df = pd.concat(parts, ignore_index=True)
    source_report = pd.DataFrame(reports)

    clean_df = pred_df[
        (~pred_df["_nn_is_outlier"].fillna(False))
        & (~pred_df["_nn_remove_source"].fillna(False))
    ].copy()

    removed_points_df = pred_df[
        pred_df["_nn_is_outlier"].fillna(False)
    ].copy()

    return (
        clean_df.reset_index(drop=True),
        source_report.reset_index(drop=True),
        removed_points_df.reset_index(drop=True),
        pred_df.reset_index(drop=True),
    )

In [ ]:
# ============================================================
# 21. Run real inference
# ============================================================

clean_df, source_report, removed_points_df, pred_df = apply_model_to_dataframe(
    model,
    df,
    threshold=best_threshold,
    max_sources=MAX_SOURCES_REAL,
    source_selection="first",
    min_points_after=20,
    max_outlier_fraction=0.50,
    show_plots=True,
    max_plots=MAX_PLOTS,
)

display(source_report)
display(removed_points_df.head())
display(clean_df.head())

In [ ]:
# ============================================================
# 22. Stats des prédictions réelles
# ============================================================

display(source_report.describe())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pred_df["_nn_anomaly_proba"].dropna(), bins=60)
ax.axvline(best_threshold, linestyle="--", label=f"threshold={best_threshold:.2f}")
ax.set_title("Distribution des probabilités d'anomalie sur vraies données")
ax.set_xlabel("NN anomaly probability")
ax.set_ylabel("count")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(source_report["outlier_fraction"].dropna(), bins=50)
ax.set_title("Fraction d'outliers prédits par source")
ax.set_xlabel("outlier fraction")
ax.set_ylabel("source count")
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(source_report["n_total"], source_report["outlier_fraction"], s=12, alpha=0.5)
ax.set_title("Outlier fraction vs n_total")
ax.set_xlabel("n_total")
ax.set_ylabel("outlier_fraction")
ax.grid(alpha=0.3)
plt.show()

display(source_report.sort_values("outlier_fraction", ascending=False).head(20))

In [ ]:
# ============================================================
# 23. XAI locale sur une source réelle suspecte
# ============================================================

def get_real_sequence_prediction_from_group(pred_df, group_string):
    # Retrouve le groupe en utilisant la colonne source_report["group"].
    mask = np.ones(len(pred_df), dtype=bool)

    # Parse simple "col=value, col=value".
    parts = group_string.split(", ")
    for part in parts:
        if "=" in part:
            col, val = part.split("=", 1)
            if col in pred_df.columns:
                # comparaison string pour robustesse
                mask &= pred_df[col].astype(str).to_numpy() == str(val)

    sub = pred_df.loc[mask].copy()

    if len(sub) == 0:
        raise ValueError("Group not found")

    t = time_to_float_days(sub[TIME_COL])
    y = pd.to_numeric(sub[Y_COL], errors="coerce").to_numpy(dtype=float)
    yerr = safe_yerr(pd.to_numeric(sub[YERR_COL], errors="coerce").to_numpy(dtype=float))

    return predict_one_sequence(model, t, y, yerr), sub


# Choisir source réelle avec au moins un outlier et proba max élevée.
candidate_report = source_report[
    (source_report["n_outliers"] > 0)
    & (source_report["removed"] == False)
].sort_values("max_proba", ascending=False)

if len(candidate_report) == 0:
    candidate_report = source_report.sort_values("max_proba", ascending=False)

chosen_group = candidate_report.iloc[0]["group"]
print("Chosen group:", chosen_group)

real_seq_pred, real_sub = get_real_sequence_prediction_from_group(pred_df, chosen_group)
xai_real = plot_xai_for_sequence(real_seq_pred, title=f"Real source XAI | {chosen_group}", threshold=best_threshold)

In [ ]:
# ============================================================
# 24. Sauvegardes
# ============================================================

OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

source_stats.to_csv(OUT_DIR / "datavis_source_stats.csv", index=False)
synthetic_features_df.to_csv(OUT_DIR / "synthetic_features_stats.csv", index=False)
threshold_sweep_df.to_csv(OUT_DIR / "nn_threshold_sweep.csv", index=False)
xai_global_df.to_csv(OUT_DIR / "nn_xai_global_feature_importance.csv", index=False)

clean_df.to_csv(OUT_DIR / "clean_df_nn_anomaly.csv", index=False)
source_report.to_csv(OUT_DIR / "source_report_nn_anomaly.csv", index=False)
removed_points_df.to_csv(OUT_DIR / "removed_points_nn_anomaly.csv", index=False)
pred_df.to_csv(OUT_DIR / "all_predictions_nn_anomaly.csv", index=False)

print("Saved outputs in:", OUT_DIR)

# Notes d'interprétation XAI

## Permutation importance

Si permuter une feature fait beaucoup baisser le F1 ou la PR-AUC, cela signifie que le modèle dépend fortement de cette feature.

Exemples attendus :
- `local_z` important : le modèle utilise fortement le désaccord local leave-one-out ;
- `local_resid_scaled` important : le modèle détecte les écarts à la médiane locale ;
- `dt_prev`, `dt_next` importants : le modèle tient compte de l'échantillonnage irrégulier.

## Gradients / Integrated Gradients

Pour un point suspect, les integrated gradients montrent quelles features ont contribué à augmenter son score d'anomalie.

Important : ce n'est pas une preuve physique. C'est une explication locale du comportement du réseau.

## Occlusion temporelle

On masque chaque point de la séquence et on mesure la baisse de probabilité du point expliqué.

Si masquer un voisin fait fortement baisser la proba d'anomalie, cela signifie que le réseau compare le point suspect à ce voisin temporel.